In [37]:
# pip install plotly --break-system-packages

In [38]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

In [39]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

In [40]:
query = '''
SELECT
    timestamp,
    COUNT(*) AS commits,
    uid
FROM
    checker
WHERE
    status = 'ready'
    AND uid LIKE 'user_%'
    AND labname = 'project1'
GROUP BY
    uid, timestamp
'''

In [41]:
df = pd.read_sql(query, conn)

In [42]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.sort_values(by=['uid','timestamp'], inplace=True)
df['cumulative_commits'] = df.groupby('uid')['commits'].cumsum()

In [43]:
df = df.sort_values(by=['timestamp'])

In [44]:
df

,timestamp,commits,uid,cumulative_commits
777,2020-04-17 05:19:02.744528,1,user_4,1
778,2020-04-17 05:22:45.549397,1,user_4,2
779,2020-04-17 05:34:24.422370,1,user_4,3
780,2020-04-17 05:43:27.773992,1,user_4,4
781,2020-04-17 05:46:32.275104,1,user_4,5
...,...,...,...,...
278,2020-05-15 10:22:39.698523,1,user_19,26
279,2020-05-15 10:22:46.248162,1,user_19,27
280,2020-05-15 10:23:18.043212,1,user_19,28
656,2020-05-15 10:38:14.430013,1,user_28,27


In [51]:
users = df['uid'].unique()
timestamps = df['timestamp'].unique()

In [ ]:
frames = []
for t in timestamps:
    frame_data = df[df['timestamp'] <= t]
    traces = []
    for user in users:
        user_data = frame_data[frame_data['uid'] == user]
        trace = go.Scatter(
            x=user_data["timestamp"],
            y=user_data["cumulative_commits"],
            mode="lines+markers",
            name=user
        )
        traces.append(trace)
    frames.append(go.Frame(data=traces, name=str(t)))

In [ ]:
initial_traces = []
for user in users:
    user_data = df[(df['uid'] == user) & (df['timestamp'] == timestamps[0])]
    trace = go.Scatter(
        x=user_data['timestamp'],
        y=user_data['cumulative_commits'],
        mode='lines+markers',
        name=user
    )
    initial_traces.append(trace)

In [ ]:
fig = go.Figure(
    data=initial_traces,
    layout=go.Layout(
        title = 'Dynamic of commits per user in project1',
        xaxis = dict(title='Timestamp'),
        yaxis = dict(title='Number of Trials'),
        updatemenus=[{
            'buttons':[
                {
                    'args': [None, {'frame': {'duration': 500, 'redraw': True},
                                    'fromcurrent': True}],
                    'label': 'Play',
                    'method': 'animate'
                },
                {
                    'args': [[None], {'frame': {'duration': 0, 'redraw': True},
                                      'mode': 'immediate',
                                      'transition': {'duration': 0}}],
                    'label': 'Pause',
                    'method': 'animate'
                }
            ],
            'direction': 'left',
            'pad': {'r': 10, 't': 87},
            'showactive': False,
            'type': 'buttons',
            'x': 0.1,
            'xanchor': 'right',
            'y': 0,
            'yanchor': 'top'
        }]
    ),
    frames=frames
)

fig.show()

In [49]:
conn.close()